In [1]:
# ============================================================
# FUCHIBOL26 - Notebook 02: Modelo Elo
# ============================================================
# El sistema Elo asigna puntos a cada selección basándose
# en sus resultados históricos. Equipos fuertes = más puntos.
# ============================================================

import pandas as pd
import os

# Cargamos los partidos oficiales que ya limpiamos antes
RUTA = os.path.join("..", "datos", "historicos", "resultados.csv")
df = pd.read_csv(RUTA)

# Limpieza rápida (igual que en el notebook anterior)
df = df.dropna(subset=["home_score", "away_score"])
df["home_score"] = df["home_score"].astype(int)
df["away_score"] = df["away_score"].astype(int)
df["date"] = pd.to_datetime(df["date"])

# Solo partidos oficiales
TORNEOS_OFICIALES = [
    "FIFA World Cup",
    "FIFA World Cup qualification",
    "UEFA Euro",
    "UEFA Euro qualification",
    "Copa América",
    "African Cup of Nations",
    "African Cup of Nations qualification",
    "AFC Asian Cup",
    "AFC Asian Cup qualification",
    "Gold Cup",
    "UEFA Nations League",
    "CONCACAF Nations League"
]

df_oficial = df[df["tournament"].isin(TORNEOS_OFICIALES)].copy()
df_oficial = df_oficial.sort_values("date").reset_index(drop=True)

print("✅ Datos cargados")
print(f"📊 Partidos oficiales: {len(df_oficial):,}")
print(f"📅 Desde: {df_oficial['date'].min().date()} → {df_oficial['date'].max().date()}")

✅ Datos cargados
📊 Partidos oficiales: 19,750
📅 Desde: 1916-07-02 → 2026-06-14


In [2]:
# Celda 2: La fórmula del sistema Elo
# ============================================================

# Todos los equipos empiezan con 1000 puntos
# Este diccionario guarda los puntos de cada selección
elo = {}

def obtener_elo(equipo):
    """Si el equipo no tiene puntos aún, le damos 1000 para empezar"""
    if equipo not in elo:
        elo[equipo] = 1000
    return elo[equipo]

def actualizar_elo(equipo_local, equipo_visit, goles_local, goles_visit, k=30):
    """
    Calcula y actualiza los puntos Elo después de un partido.
    
    Parámetros:
        equipo_local: nombre del equipo de casa
        equipo_visit: nombre del equipo visitante
        goles_local:  goles que metió el local
        goles_visit:  goles que metió el visitante
        k:            qué tan rápido cambian los puntos (usamos 30)
    """
    # Paso 1: Obtenemos los puntos actuales de cada equipo
    elo_local = obtener_elo(equipo_local)
    elo_visit = obtener_elo(equipo_visit)

    # Paso 2: Calculamos qué resultado se esperaba
    # Esta fórmula viene del sistema Elo original
    esperado_local = 1 / (1 + 10 ** ((elo_visit - elo_local) / 400))
    esperado_visit = 1 - esperado_local

    # Paso 3: ¿Qué pasó realmente?
    if goles_local > goles_visit:
        real_local = 1      # Ganó el local
        real_visit = 0
    elif goles_local < goles_visit:
        real_local = 0      # Ganó el visitante
        real_visit = 1
    else:
        real_local = 0.5    # Empate
        real_visit = 0.5

    # Paso 4: Actualizamos los puntos
    elo[equipo_local] = elo_local + k * (real_local - esperado_local)
    elo[equipo_visit] = elo_visit + k * (real_visit - esperado_visit)

# Procesamos TODOS los partidos históricos en orden cronológico
print("⏳ Procesando partidos históricos...")

for _, partido in df_oficial.iterrows():
    actualizar_elo(
        partido["home_team"],
        partido["away_team"],
        partido["home_score"],
        partido["away_score"]
    )

print(f"✅ {len(df_oficial):,} partidos procesados")
print(f"🌍 {len(elo)} selecciones con puntuación Elo")

⏳ Procesando partidos históricos...
✅ 19,750 partidos procesados
🌍 223 selecciones con puntuación Elo


In [3]:
# Celda 3: Ver el ranking Elo final
# Convertimos el diccionario en una tabla ordenada

df_elo = pd.DataFrame([
    {"seleccion": equipo, "puntos_elo": round(puntos)}
    for equipo, puntos in elo.items()
])

# Ordenamos de mayor a menor puntuación
df_elo = df_elo.sort_values("puntos_elo", ascending=False)
df_elo = df_elo.reset_index(drop=True)
df_elo.index += 1  # Ranking desde 1

print("🏆 RANKING ELO MUNDIAL — TOP 20")
print("="*35)
print(df_elo.head(20).to_string())

print(f"\n📊 Puntuación más alta: {df_elo['puntos_elo'].max()}")
print(f"📊 Puntuación más baja: {df_elo['puntos_elo'].min()}")
print(f"📊 Promedio mundial:    {df_elo['puntos_elo'].mean():.0f}")

🏆 RANKING ELO MUNDIAL — TOP 20
        seleccion  puntos_elo
1           Spain        1457
2          France        1388
3       Argentina        1363
4         England        1362
5           Japan        1350
6          Mexico        1344
7            Iran        1338
8        Portugal        1332
9         Morocco        1328
10        Germany        1325
11      Australia        1317
12    South Korea        1315
13    Netherlands        1309
14        Senegal        1305
15  United States        1303
16         Brazil        1291
17     Uzbekistan        1275
18          Italy        1270
19        Croatia        1269
20         Panama        1268

📊 Puntuación más alta: 1457
📊 Puntuación más baja: 447
📊 Promedio mundial:    1000


In [4]:
# Celda 4: Buscar cualquier selección en el ranking

SELECCIONES = ["Mexico", "Guatemala", "Brazil", "France", "Spain", "Argentina"]

print("🔍 POSICIÓN EN EL RANKING ELO")
print("="*45)

for nombre in SELECCIONES:
    # Buscamos la posición en la tabla
    fila = df_elo[df_elo["seleccion"] == nombre]
    
    if len(fila) > 0:
        posicion = fila.index[0]
        puntos = fila["puntos_elo"].values[0]
        print(f"  #{posicion:<4} {nombre:<15} {puntos} puntos")
    else:
        print(f"  ❌ {nombre} no encontrado")

🔍 POSICIÓN EN EL RANKING ELO
  #6    Mexico          1344 puntos
  #57   Guatemala       1130 puntos
  #16   Brazil          1291 puntos
  #2    France          1388 puntos
  #1    Spain           1457 puntos
  #3    Argentina       1363 puntos


In [5]:
# Celda 5: Guardar el ranking Elo en un archivo CSV

RUTA_ELO = os.path.join("..", "datos", "procesados", "ranking_elo.csv")

df_elo.to_csv(RUTA_ELO, index=True, index_label="ranking")

print("✅ Ranking Elo guardado")
print(f"📁 Ubicación: datos/procesados/ranking_elo.csv")
print(f"🌍 Selecciones guardadas: {len(df_elo)}")
print(f"\n🏆 Resumen del Modelo Elo:")
print(f"   Partidos procesados: {len(df_oficial):,}")
print(f"   Puntuación más alta: {df_elo['puntos_elo'].max()} ({df_elo.iloc[0]['seleccion']})")
print(f"   Mexico está en el puesto: #{df_elo[df_elo['seleccion']=='Mexico'].index[0]}")
print(f"   Guatemala está en el puesto: #{df_elo[df_elo['seleccion']=='Guatemala'].index[0]}")

✅ Ranking Elo guardado
📁 Ubicación: datos/procesados/ranking_elo.csv
🌍 Selecciones guardadas: 223

🏆 Resumen del Modelo Elo:
   Partidos procesados: 19,750
   Puntuación más alta: 1457 (Spain)
   Mexico está en el puesto: #6
   Guatemala está en el puesto: #57
